In [1]:
import os
import pandas as pd
import numpy as np

# Ensure relative paths work regardless of execution folder
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
RAW_DATA_PATH = os.path.join(PROJECT_ROOT, "data", "raw")
PROCESSED_DATA_PATH = os.path.join(PROJECT_ROOT, "data", "processed")

os.makedirs(PROCESSED_DATA_PATH, exist_ok=True)
print(f"Project Root: {PROJECT_ROOT}")

ModuleNotFoundError: No module named 'pandas'

In [ ]:
# 1. World Bank Global Findex Data (2011–2024)
findex_raw = pd.DataFrame({
    'Year': [2011, 2014, 2017, 2021, 2024],
    'Account_Ownership_Total_Pct': [14.0, 22.0, 35.0, 46.0, 49.0],
    'Account_Ownership_Male_Pct': [17.1, 26.3, 41.2, 55.8, 55.8],
    'Account_Ownership_Female_Pct': [11.0, 17.8, 29.1, 36.4, 42.1],
    'Mobile_Money_Account_Pct': [0.0, 0.1, 0.3, 4.7, 10.2]
})

# 2. Macroeconomic & Infrastructure Indicators (Annual 2011–2024)
years = list(range(2011, 2025))
macro_raw = pd.DataFrame({
    'Year': years,
    '4G_Coverage_Pct': [0.0, 0.0, 2.0, 5.0, 12.0, 20.0, 35.0, 48.0, 62.0, 75.0, 82.0, 88.0, 93.0, 96.0],
    'Mobile_Subscriptions_Per_100': [14.8, 23.1, 30.5, 38.2, 46.7, 51.0, 56.4, 58.9, 63.2, 65.0, 68.1, 71.4, 75.8, 80.2]
})

print("--- Findex Raw Data (.head()) ---")
display(findex_raw.head())

print("\n--- Macro Indicators Raw Data (.head()) ---")
display(macro_raw.head())

In [ ]:
print("=== Findex Dataset Info ===")
findex_raw.info()

print("\n=== Macro Dataset Info ===")
macro_raw.info()

In [ ]:
print("=== Findex Dataset Info ===")
findex_raw.info()

print("\n=== Macro Dataset Info ===")
macro_raw.info()

In [ ]:
# 1. Cubic spline interpolation for sparse Findex survey years (2011-2024)
findex_cols = [
    'Account_Ownership_Total_Pct', 
    'Account_Ownership_Male_Pct', 
    'Account_Ownership_Female_Pct', 
    'Mobile_Money_Account_Pct'
]

for col in findex_cols:
    df_unified[f'{col}_Interpolated'] = df_unified[col].interpolate(method='pchip').round(2)

# 2. Feature Engineering: Gender Equity Gap (Male % - Female %)
df_unified['Gender_Access_Gap_pp'] = (
    df_unified['Account_Ownership_Male_Pct_Interpolated'] - 
    df_unified['Account_Ownership_Female_Pct_Interpolated']
).round(2)

# 3. Feature Engineering: Annual Growth Rates
df_unified['Account_Ownership_YoY_pp'] = df_unified['Account_Ownership_Total_Pct_Interpolated'].diff().round(2)

print("--- Cleaned and Imputed Unified Dataset (.head()) ---")
display(df_unified.head(10))

In [ ]:
# Validate no unexpected NaN values remain in core analysis columns
assert df_unified['Account_Ownership_Total_Pct_Interpolated'].isna().sum() == 0, "Null values remain in target variable!"

# Save processed dataset
output_file = os.path.join(PROCESSED_DATA_PATH, "ethiopia_fi_unified_data.csv")
df_unified.to_csv(output_file, index=False)

print(f"SUCCESS: Unified data processed and exported to {output_file}")
print("\nFinal Dataset Summary Statistics:")
display(df_unified.describe().round(2))